In [ ]:
# ============================================================
# BIASCOPE — Notebook 05 (CORRECTED, single cell)
# Hindi + Hinglish bias-dataset preparation
# Phase 1: candidate selection + annotation sheets
# Phase 2 (annotation ke baad, same cell dobara run): IAA + resolution + final dataset
# ============================================================
import json, re, sys, subprocess, unicodedata, warnings
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

# ---------------- Config ----------------
SEED = 42
N_SENTENCES = 1500             # unique sentences (har ek ke 4 aligned versions)
PILOT_SIZE = 100               # sabhi annotators pehle ye annotate karein
ANNOTATORS = ["A", "B"]        # 3rd annotator ho to "C" add karo
OVERLAP_FRAC = 0.30            # non-pilot items ka hissa jo sabhi annotators karein (IAA ke liye)
VERIFY_SIZE = 100              # Hinglish label verification sample
CMI_MIN, CMI_MAX = 0.10, 0.90  # hicm me genuine code-mixing
MIN_WORDS, MAX_WORDS = 8, 60
MIN_BIASED_RATE_PILOT = 0.05   # pilot gate
FORCE_REBUILD = False          # True karoge to assignment dobara banega (annotation ke baad kabhi mat karna)

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
MULTILINGUAL_DIR = BASE_DIR / "data" / "multilingual"
ANNOTATION_DIR = MULTILINGUAL_DIR / "annotation"
SHEETS_DIR = ANNOTATION_DIR / "sheets"          # annotators ko do
COMPLETED_DIR = ANNOTATION_DIR / "completed"    # annotators se wapas
FINAL_DIR = MULTILINGUAL_DIR / "final"
for d in (SHEETS_DIR, COMPLETED_DIR, FINAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

HINMIX_PATH = MULTILINGUAL_DIR / "hinmix_clean_long.csv"
MASTER_PATH = ANNOTATION_DIR / "assignment_master.csv"

TEXT_COLS = ["text_en", "text_hi", "text_hicm", "text_hicmrom"]
VERSION_MAP = {"text_hi": "hindi", "text_hicm": "hinglish_devanagari",
               "text_hicmrom": "hinglish_romanized", "text_en": "english"}
LANG_MAP = {"hi": "hindi", "hicm": "hinglish_devanagari",
            "hicmrom": "hinglish_romanized", "en": "english"}
DIMS = ["subjectivity", "sensationalism", "loaded_language",
        "political_framing", "emotional_tone"]
ANN_COLS = ["bias_label"] + DIMS + ["biased_span", "usable", "confidence", "notes"]

# ---------------- Helpers ----------------
DEV = r"\u0900-\u097F"
_ZW_ALWAYS = re.compile(r"[\u200b\u200e\u200f\u202a-\u202e\u2060\ufeff]")
_ZWJ_ORPHAN = re.compile(rf"(?<![{DEV}])[\u200c\u200d]|[\u200c\u200d](?![{DEV}])")
_BRACKETS = {"LRB": "(", "RRB": ")", "LSB": "[", "RSB": "]", "LCB": "{", "RCB": "}"}
_BRACKET_RE = re.compile(r"\b(?:LRB|RRB|LSB|RSB|LCB|RCB)\b")

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = _BRACKET_RE.sub(lambda m: _BRACKETS[m.group(0)], text)
    text = _ZW_ALWAYS.sub("", text)
    text = _ZWJ_ORPHAN.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"([\(\[\{])\s+", r"\1", text)
    text = re.sub(r"\s+([\)\]\}])", r"\1", text)
    return text

def cmi(t):
    toks = t.split()
    dev = sum(bool(re.search(rf"[{DEV}]", w)) for w in toks)
    lat = sum(bool(re.search(r"[A-Za-z]", w)) for w in toks)
    return lat / (dev + lat) if (dev + lat) else 0.0

def digit_ratio(t):
    s = re.sub(r"\s", "", t)
    return sum(c.isdigit() for c in s) / len(s) if s else 1.0

NON_NEWS_RE = re.compile(
    r"\b(plugin|menu|toolbar|dialog|click|button|desktop|folder|file|section|"
    r"subsection|clause|chapter|figure|table|schedule)\b", re.IGNORECASE)

ANNOTATION_GUIDELINES = """
BIASCOPE Hindi & Hinglish Annotation Guidelines
================================================
Task: sentence me meaningful media/news bias hai ya nahi.

bias_label: 0 = Neutral, 1 = Biased
Biased tab jab sentence me inme se ek ya zyada ho:
  1. Loaded language   -> charged/judgmental words jo reader ki perception influence karein
  2. Sensationalism    -> exaggeration, alarmist ya attention-seeking framing
  3. Political framing -> kisi political actor/group ke favour ya against one-sided framing
  4. Emotional framing -> perception influence karne wali strong emotional language
  5. Subjective eval.  -> opinion ko fact ki tarah present karna, explicit judgment

Dimension columns (subjectivity, sensationalism, loaded_language,
political_framing, emotional_tone): 1 = present, 0 = absent.
Agar bias_label = 0 hai to sab dimensions 0 rakho.

biased_span: sentence ka wo phrase (copy-paste) jo bias dikhata hai. Neutral me khali chhodo.
usable: 1 = theek sentence. 0 = fragment / UI string / numbered clause /
        meaningless / news-like nahi (aise rows final dataset se hatenge).
confidence: 1 = low, 2 = medium, 3 = high.

Sirf is wajah se BIASED mat likho ki sentence:
  - politics discuss karta hai
  - negative event describe karta hai
  - factual reporting ke liye zaroori emotional words use karta hai
  - Hinglish ya code-switched hai
Label linguistic framing pe depend kare, language pe nahi.

Primary text: text_hi. text_en sirf reference ke liye hai (disambiguation).
Pehli 100 rows (pilot) pehle annotate karo, aur discuss karne ke baad aage badho.
"""

# ============================================================
# PHASE 1 — candidate selection + annotation sheets
# ============================================================
if MASTER_PATH.exists() and not FORCE_REBUILD:
    master = pd.read_csv(MASTER_PATH, encoding="utf-8")
    print(f"Frozen assignment mila ({len(master)} sentences) — reuse kar raha hoon, sheets dobara nahi banenge.")
else:
    assert HINMIX_PATH.exists(), "Pehle Notebook 04 chalao (hinmix_clean_long.csv missing)"
    hinmix = pd.read_csv(HINMIX_PATH)
    assert hinmix.duplicated(["row_id", "lang"]).sum() == 0, "row_id/lang duplicates hain"

    base_col = "normalized_text" if "normalized_text" in hinmix.columns else "text"
    hinmix["text_normalized"] = hinmix[base_col].map(normalize_text)   # idempotent
    hinmix["language_category"] = hinmix["lang"].map(LANG_MAP)
    assert hinmix["language_category"].notna().all(), "unknown lang value"

    # wide format: ek row_id = ek sentence ke 4 aligned versions
    wide = (hinmix.pivot(index="row_id", columns="lang", values="text_normalized")
                  .rename(columns={"en": "text_en", "hi": "text_hi",
                                   "hicm": "text_hicm", "hicmrom": "text_hicmrom"})
                  .dropna().reset_index())
    print("Aligned sentences (4 versions each):", len(wide))

    wide["cmi_hicm"] = wide["text_hicm"].map(cmi)
    wide["n_words_hi"] = wide["text_hi"].str.split().str.len()

    # ---- screening: legal/UI/numbered fragments hatao ----
    screen = pd.DataFrame({
        "length_ok":    wide["n_words_hi"].between(MIN_WORDS, MAX_WORDS),
        "digits_ok":    wide["text_hi"].map(digit_ratio) <= 0.05,
        "not_ui_legal": ~wide["text_en"].str.contains(NON_NEWS_RE),
        "not_numbered": ~wide["text_en"].str.match(r"^\s*\d+\b"),
        "cmi_ok":       wide["cmi_hicm"].between(CMI_MIN, CMI_MAX),
    })
    print("\nScreening — filter-wise fails:")
    for c in screen.columns:
        print(f"  {c:13s}: {(~screen[c]).sum()}")
    pool = wide[screen.all(axis=1)].drop_duplicates("text_hi").reset_index(drop=True)
    print(f"Pool after screening: {len(pool)} / {len(wide)}")
    assert len(pool) >= PILOT_SIZE + 200, "Pool bahut chhota — thresholds loosen karo"

    n = min(N_SENTENCES, len(pool))
    sel = pool.sample(n=n, random_state=SEED).reset_index(drop=True)

    print("\nEyeball check — 10 random candidates (kya ye news-like lagte hain?):")
    display(sel[["row_id", "text_hi", "text_en"]].head(10))

    # ---- roles: pilot (sabke liye) / overlap (sabke liye) / exclusive (split) ----
    pilot = sel.iloc[:PILOT_SIZE]
    rest = sel.iloc[PILOT_SIZE:]
    n_overlap = int(len(rest) * OVERLAP_FRAC)
    overlap = rest.iloc[:n_overlap]
    exclusive = rest.iloc[n_overlap:]
    chunks = np.array_split(np.arange(len(exclusive)), len(ANNOTATORS))

    parts = [pilot.assign(role="pilot", annotators="|".join(ANNOTATORS)),
             overlap.assign(role="overlap", annotators="|".join(ANNOTATORS))]
    for a, ch in zip(ANNOTATORS, chunks):
        parts.append(exclusive.iloc[ch].assign(role="exclusive", annotators=a))
    master = pd.concat(parts, ignore_index=True)

    verify = rest.sample(n=min(VERIFY_SIZE, len(rest)), random_state=SEED)
    master["is_verify"] = master["row_id"].isin(verify["row_id"])
    master.to_csv(MASTER_PATH, index=False, encoding="utf-8")

    # ---- annotator sheets (sirf row_id, text_hi, text_en dikhte hain) ----
    for i, a in enumerate(ANNOTATORS):
        rest_part = (pd.concat([overlap, exclusive.iloc[chunks[i]]])
                       .sample(frac=1, random_state=SEED + i))
        sheet = pd.concat([pilot, rest_part])[["row_id", "text_hi", "text_en"]].reset_index(drop=True)
        for c in ANN_COLS:
            sheet[c] = ""
        sheet["usable"] = 1
        sheet.to_csv(SHEETS_DIR / f"annotator_{a}.csv", index=False, encoding="utf-8-sig")
        print(f"Sheet saved: annotator_{a}.csv  ({len(sheet)} rows)")

    # ---- Hinglish verification sheet (projected label check ke liye) ----
    vsheet = verify[["row_id", "text_hicm", "text_hicmrom"]].copy()
    vsheet["label_hicm"] = ""
    vsheet["label_hicmrom"] = ""
    vsheet.to_csv(SHEETS_DIR / "hinglish_verification.csv", index=False, encoding="utf-8-sig")

    (ANNOTATION_DIR / "annotation_guidelines.txt").write_text(ANNOTATION_GUIDELINES, encoding="utf-8")

    metadata = {
        "project": "BIASCOPE", "notebook": "05", "seed": SEED,
        "hinmix_source": "kartikagg98/HINMIX_hi-en", "hinmix_is_bias_labelled": False,
        "unique_sentences": int(len(master)), "versions_per_sentence": 4,
        "annotators": ANNOTATORS, "pilot_size": PILOT_SIZE,
        "overlap_frac": OVERLAP_FRAC, "verify_size": VERIFY_SIZE,
        "screening": {"min_words": MIN_WORDS, "max_words": MAX_WORDS,
                      "cmi_range": [CMI_MIN, CMI_MAX]},
        "label_strategy": "annotate Hindi once; project label to aligned versions via row_id; "
                          "verify projection on Hinglish sample",
        "bias_labels": {"0": "neutral", "1": "biased"},
        "annotation_status": "human_annotation_required",
    }
    with open(ANNOTATION_DIR / "notebook_05_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=4)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    master["role"].value_counts().plot(kind="bar", ax=ax[0], title="Sentences by role")
    ax[1].hist(master["n_words_hi"], bins=40)
    ax[1].set_title("Hindi word length (candidates)")
    plt.tight_layout(); plt.show()

    print(f"\nPHASE 1 DONE. Sheets: {SHEETS_DIR}")
    print("Annotators sheets Google Sheets/Excel me bharein, phir CSV (UTF-8) me save karke")
    print(f"{COMPLETED_DIR} me 'annotator_<ID>.csv' naam se daalein, aur is cell ko dobara chalao.")

# ============================================================
# PHASE 2 — annotation ke baad: validation, IAA, resolution, final dataset
# ============================================================
frames = []
for a in ANNOTATORS:
    p = COMPLETED_DIR / f"annotator_{a}.csv"
    if p.exists():
        df = pd.read_csv(p, encoding="utf-8-sig")
        df["annotator_id"] = a
        frames.append(df)

ann = pd.concat(frames, ignore_index=True) if frames else None
if ann is not None:
    ann["bias_label"] = pd.to_numeric(ann["bias_label"], errors="coerce")
    ann = ann[ann["bias_label"].notna()].copy()

if ann is None or len(ann) == 0:
    print("\nAbhi koi completed annotation nahi mila — Phase 2 skip.")
else:
    print("\n" + "=" * 60 + "\nPHASE 2 — ANALYSIS\n" + "=" * 60)

    # ---- validation ----
    bad = ~ann["bias_label"].isin([0, 1])
    print("Invalid bias_label rows (dropped):", int(bad.sum()))
    ann = ann[~bad].drop_duplicates(["row_id", "annotator_id"]).copy()
    ann["usable"] = pd.to_numeric(ann["usable"], errors="coerce").fillna(1)
    ann["confidence"] = pd.to_numeric(ann["confidence"], errors="coerce")
    for c in DIMS:
        ann[c] = pd.to_numeric(ann[c], errors="coerce").fillna(0)
        ann.loc[~ann[c].isin([0, 1]), c] = 0
    print("Annotated rows per annotator:\n", ann["annotator_id"].value_counts().to_string())

    # ---- pilot gate ----
    pilot_ids = set(master.loc[master["role"] == "pilot", "row_id"])
    pil = ann[ann["row_id"].isin(pilot_ids)]
    if len(pil):
        print("\n--- PILOT GATE ---")
        print(pil.groupby("annotator_id").agg(
            n=("bias_label", "size"),
            biased_rate=("bias_label", "mean"),
            unusable_rate=("usable", lambda s: 1 - s.mean())).round(3).to_string())
        rate = pil.loc[pil["usable"] == 1, "bias_label"].mean()
        if rate < MIN_BIASED_RATE_PILOT:
            print(f"\nWARNING: pilot biased rate {rate:.1%} < {MIN_BIASED_RATE_PILOT:.0%}.")
            print("HINMIX is not suitable as a bias annotation pool. Switch to scraped Hindi news ")
            print("+ label projection from MBIB (translation + human verification).")
        else:
            print(f"\nPilot biased rate {rate:.1%} — annotation pool workable hai.")

    # ---- inter-annotator agreement ----
    def pair_kappas(df, col):
        piv = df.pivot_table(index="row_id", columns="annotator_id", values=col, aggfunc="first")
        out = {}
        for a, b in combinations(piv.columns, 2):
            both = piv[[a, b]].dropna()
            if len(both) >= 10:
                try:
                    out[f"{a}-{b}"] = (round(float(cohen_kappa_score(both[a].astype(int),
                                                                     both[b].astype(int))), 3), len(both))
                except Exception:
                    pass
        return out, piv

    print("\n--- INTER-ANNOTATOR AGREEMENT (Cohen's kappa, (kappa, n_items)) ---")
    for col in ["bias_label"] + DIMS:
        k, piv_col = pair_kappas(ann, col)
        print(f"{col:18s}", k if k else "overlap kam hai (<10 items)")
        if col == "bias_label":
            piv_label = piv_col
    print("Rough guide: <0.40 weak (guidelines revise karo), 0.40-0.60 moderate, >0.60 good")

    try:
        try:
            import krippendorff
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "krippendorff"], check=True)
            import krippendorff
        multi = piv_label[piv_label.notna().sum(axis=1) >= 2]
        if len(multi) >= 10:
            alpha = krippendorff.alpha(reliability_data=multi.T.values.astype(float),
                                       level_of_measurement="nominal")
            print(f"Krippendorff alpha (bias_label, nominal): {alpha:.3f}  (items={len(multi)})")
    except Exception as e:
        print("Krippendorff skip:", e)

    # ---- aggregation / resolution ----
    g = ann.groupby("row_id")
    res = pd.DataFrame({
        "n_annotations": g.size(),
        "votes_biased": g["bias_label"].sum(),
        "usable_mean": g["usable"].mean(),
        "confidence": g["confidence"].mean(),
        "biased_span": g["biased_span"].first(),
    })
    res["bias_label"] = np.where(res["votes_biased"] * 2 > res["n_annotations"], 1.0,
                          np.where(res["votes_biased"] * 2 < res["n_annotations"], 0.0, np.nan))
    for c in DIMS:
        res[c] = (g[c].mean() > 0.5).astype(int)
    res["usable"] = (res["usable_mean"] >= 0.5).astype(int)

    expected = master.set_index("row_id")["annotators"].str.count(r"\|").add(1).rename("expected")
    res = res.join(expected, how="inner")
    pending = res[res["n_annotations"] < res["expected"]]
    res = res[res["n_annotations"] >= res["expected"]].copy()
    print(f"\nFully annotated sentences: {len(res)} | pending (kisi annotator ka baaki): {len(pending)}")

    adj_path = COMPLETED_DIR / "adjudication_resolved.csv"
    if adj_path.exists():
        r = pd.read_csv(adj_path, encoding="utf-8-sig")
        r["bias_label"] = pd.to_numeric(r["bias_label"], errors="coerce")
        r = r.dropna(subset=["bias_label"]).drop_duplicates("row_id").set_index("row_id")["bias_label"]
        res["bias_label"] = res["bias_label"].fillna(pd.Series(res.index.map(r), index=res.index))

    ties = res[res["bias_label"].isna()]
    if len(ties):
        piv_t = (ann[ann["row_id"].isin(ties.index)]
                 .pivot_table(index="row_id", columns="annotator_id", values="bias_label", aggfunc="first")
                 .add_prefix("label_"))
        adj = master.drop_duplicates("row_id").set_index("row_id").loc[ties.index, ["text_hi", "text_en"]].join(piv_t)
        adj["bias_label"] = ""
        adj.reset_index().to_csv(ANNOTATION_DIR / "adjudication_needed.csv", index=False, encoding="utf-8-sig")
        print(f"Disagreements/ties: {len(ties)} -> adjudication_needed.csv "
              f"(bias_label bharke COMPLETED_DIR me 'adjudication_resolved.csv' naam se daalo)")

    final = res[(res["usable"] == 1) & res["bias_label"].notna()].copy()
    final["bias_label"] = final["bias_label"].astype(int)
    print(f"Unusable (dropped): {int((res['usable'] == 0).sum())} | "
          f"unresolved ties: {int(res['bias_label'].isna().sum())} | FINAL sentences: {len(final)}")
    if len(final):
        print("\nLabel distribution:\n", final["bias_label"].value_counts().sort_index().to_string())
        print("Biased rate:", round(final["bias_label"].mean(), 3))
        print("Dimension prevalence:\n", final[DIMS].mean().round(3).to_string())

    # ---- Hinglish verification (projected label vs direct label) ----
    vp = COMPLETED_DIR / "hinglish_verification.csv"
    if vp.exists() and len(final):
        v = pd.read_csv(vp, encoding="utf-8-sig").merge(
            final[["bias_label"]].reset_index(), on="row_id", how="inner")
        print("\n--- HINGLISH PROJECTION CHECK ---")
        for col in ["label_hicm", "label_hicmrom"]:
            x = pd.to_numeric(v[col], errors="coerce")
            m = x.notna()
            if m.sum() >= 10:
                agree = (x[m] == v.loc[m, "bias_label"]).mean()
                kap = cohen_kappa_score(v.loc[m, "bias_label"].astype(int), x[m].astype(int))
                print(f"{col:14s} agreement={agree:.1%}  kappa={kap:.3f}  n={int(m.sum())}")

    # ---- group-wise split + export ----
    if len(final) >= 50:
        ids = final.index.to_numpy()
        y = final["bias_label"].to_numpy()
        strat = y if np.bincount(y, minlength=2).min() >= 10 else None
        tr_ids, tmp_ids, _, y_tmp = train_test_split(ids, y, test_size=0.2, random_state=SEED, stratify=strat)
        strat2 = y_tmp if np.bincount(y_tmp, minlength=2).min() >= 2 else None
        va_ids, te_ids = train_test_split(tmp_ids, test_size=0.5, random_state=SEED, stratify=strat2)
        split_map = {**{i: "train" for i in tr_ids}, **{i: "val" for i in va_ids}, **{i: "test" for i in te_ids}}

        fw = (master.drop_duplicates("row_id").set_index("row_id")
                    .loc[final.index, TEXT_COLS + ["cmi_hicm"]]
                    .join(final[["bias_label"] + DIMS + ["biased_span", "confidence", "n_annotations"]]))
        rows = []
        for col, cat in VERSION_MAP.items():
            tmp = fw[[col, "bias_label"] + DIMS + ["biased_span"]].reset_index().rename(columns={col: "text"})
            tmp["language_category"] = cat
            rows.append(tmp)
        long_df = pd.concat(rows, ignore_index=True)
        long_df["split"] = long_df["row_id"].map(split_map)

        long_df.to_csv(FINAL_DIR / "biascope_hx_long.csv", index=False, encoding="utf-8")
        fw.assign(split=fw.index.map(split_map)).reset_index().to_csv(
            FINAL_DIR / "biascope_hx_wide.csv", index=False, encoding="utf-8")
        for s in ["train", "val", "test"]:
            long_df[long_df["split"] == s].to_csv(FINAL_DIR / f"biascope_hx_{s}.csv", index=False, encoding="utf-8")

        print("\nSplit (sentences, row_id-level, no leakage):",
              {s: sum(1 for v_ in split_map.values() if v_ == s) for s in ["train", "val", "test"]})
        print("Exported to:", FINAL_DIR)
    else:
        print("\nFinal sentences < 50 — split/export abhi skip (aur annotate karo).")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 05: Annotation Pre-flight Validation
# Read-only: assignment rebuild nahi karta, labels modify nahi karta
# ============================================================
import re
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
ANNOTATION_DIR = BASE_DIR / "data" / "multilingual" / "annotation"
COMPLETED_DIR = ANNOTATION_DIR / "completed"
MASTER_PATH = ANNOTATION_DIR / "assignment_master.csv"
ANNOTATORS = ["A", "B"]
MIN_BIASED_RATE_PILOT = 0.05

DIMS = ["subjectivity", "sensationalism", "loaded_language",
        "political_framing", "emotional_tone"]
REQUIRED_COLS = (["row_id", "text_hi", "text_en", "bias_label"] + DIMS +
                 ["biased_span", "usable", "confidence", "notes"])

def num(s):
    return pd.to_numeric(s, errors="coerce")

assert MASTER_PATH.exists(), "assignment_master.csv missing — pehle Phase 1 chalao"
master = pd.read_csv(MASTER_PATH, encoding="utf-8")

summary, pilot_labels = {}, {}

for a in ANNOTATORS:
    print("\n" + "=" * 65 + f"\nAnnotator {a}\n" + "=" * 65)

    exp_df = master[master["annotators"].str.split("|").map(lambda lst: a in lst)]
    exp_ids = set(exp_df["row_id"])
    pilot_ids = set(exp_df.loc[exp_df["role"] == "pilot", "row_id"])
    print(f"Expected rows: {len(exp_ids)}  (pilot: {len(pilot_ids)})")

    p = COMPLETED_DIR / f"annotator_{a}.csv"
    if not p.exists():
        print(f"⏳ Completed file nahi mila: {p}")
        summary[a] = {"status": "missing"}
        continue

    try:
        df = pd.read_csv(p, encoding="utf-8-sig")
    except UnicodeDecodeError:
        print("❌ File UTF-8 nahi hai. Sheets/Excel se 'CSV UTF-8' me dobara save karo.")
        summary[a] = {"status": "bad_encoding"}
        continue

    missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing_cols:
        print("❌ Missing columns:", missing_cols)
        summary[a] = {"status": "bad_columns"}
        continue
    print(f"✓ File loaded: {len(df)} rows, saare columns present")

    # ---- encoding sanity (Devanagari preserved?) ----
    dev_share = df["text_hi"].astype(str).map(
        lambda t: len(re.findall(r"[\u0900-\u097F]", t)) / max(len(t), 1)).mean()
    if dev_share < 0.5:
        print(f"❌ text_hi me Devanagari share sirf {dev_share:.0%} — encoding kharab lag rahi hai")
    else:
        print(f"✓ Devanagari text intact ({dev_share:.0%} chars)")

    # ---- row_id vs assignment ----
    ids = set(df["row_id"])
    missing_ids = exp_ids - ids
    extra_ids = ids - exp_ids
    dup_ids = int(df["row_id"].duplicated().sum())
    print(f"Row check -> missing: {len(missing_ids)} | extra: {len(extra_ids)} | duplicates: {dup_ids}")

    # ---- label validity ----
    y = num(df["bias_label"])
    valid = y.isin([0, 1])
    blank = y.isna()
    invalid = (~valid) & (~blank)
    done_ids = set(df.loc[valid, "row_id"]) & exp_ids
    print(f"Labelled: {int(valid.sum())} | blank: {int(blank.sum())} | invalid: {int(invalid.sum())}")
    print(f"Progress (expected rows labelled): {len(done_ids)}/{len(exp_ids)} "
          f"= {len(done_ids) / max(len(exp_ids), 1):.1%}")

    if valid.sum():
        print("Distribution:", y[valid].value_counts().sort_index()
              .rename(index={0: "Neutral", 1: "Biased"}).to_dict())

    # ---- other columns ----
    bad_dims = {c: int((~num(df[c]).isin([0, 1]) & num(df[c]).notna()).sum()) for c in DIMS}
    bad_dims = {c: n for c, n in bad_dims.items() if n}
    cf, u = num(df["confidence"]), num(df["usable"])
    bad_cf = int((~cf.isin([1, 2, 3]) & cf.notna()).sum())
    bad_u = int((~u.isin([0, 1]) & u.notna()).sum())
    if bad_dims: print("⚠ Invalid dimension values:", bad_dims)
    if bad_cf:   print("⚠ Invalid confidence values (1/2/3 hona chahiye):", bad_cf)
    if bad_u:    print("⚠ Invalid usable values (0/1 hona chahiye):", bad_u)
    print("Unusable rows (usable=0):", int((u == 0).sum()))

    # ---- label consistency ----
    lab = df[valid].copy()
    lab["bias_label"] = y[valid]
    for c in DIMS:
        lab[c] = num(lab[c]).fillna(0)
    dim_sum = lab[DIMS].sum(axis=1)
    n_neutral_dim = int(((lab["bias_label"] == 0) & (dim_sum > 0)).sum())
    n_biased_nodim = int(((lab["bias_label"] == 1) & (dim_sum == 0)).sum())
    n_biased_nospan = int(((lab["bias_label"] == 1) & lab["biased_span"].isna()).sum())
    if n_neutral_dim:   print(f"⚠ Neutral par dimension=1: {n_neutral_dim} rows")
    if n_biased_nodim:  print(f"⚠ Biased par saari dimensions 0: {n_biased_nodim} rows")
    if n_biased_nospan: print(f"⚠ Biased par biased_span khali: {n_biased_nospan} rows")

    # ---- pilot ----
    pil = df[df["row_id"].isin(pilot_ids)]
    py = num(pil["bias_label"])
    pil_done = int(py.isin([0, 1]).sum())
    pil_use = num(pil["usable"]).fillna(1)
    pil_rate = py[py.isin([0, 1]) & (pil_use == 1)].mean()
    print(f"\nPilot: {pil_done}/{len(pilot_ids)} labelled"
          + (f" | biased rate (usable): {pil_rate:.1%}" if pil_done else ""))
    pilot_labels[a] = pil.set_index("row_id")["bias_label"].pipe(num)

    clean = (not missing_cols and not len(missing_ids) and not len(extra_ids) and not dup_ids
             and not int(invalid.sum()) and not bad_dims and not bad_cf and not bad_u)
    summary[a] = {
        "status": "loaded", "clean": bool(clean),
        "pilot_done": pil_done, "pilot_total": len(pilot_ids),
        "all_done": len(done_ids) == len(exp_ids),
        "pilot_rate": pil_rate,
    }

# ---- pilot agreement preview (raw, kappa Phase 2 me aayega) ----
loaded = [a for a in ANNOTATORS if summary.get(a, {}).get("status") == "loaded"]
if len(loaded) >= 2:
    print("\n" + "=" * 65 + "\nPILOT RAW AGREEMENT\n" + "=" * 65)
    for a, b in combinations(loaded, 2):
        both = pd.concat([pilot_labels[a], pilot_labels[b]], axis=1, keys=[a, b]).dropna()
        both = both[both[a].isin([0, 1]) & both[b].isin([0, 1])]
        if len(both):
            print(f"{a}-{b}: {(both[a] == both[b]).mean():.1%} agreement on {len(both)} pilot rows")

# ---- readiness ----
print("\n" + "=" * 65 + "\nREADINESS\n" + "=" * 65)
if not loaded:
    print("⏳ NOT READY — completed files abhi nahi aaye.")
else:
    pilot_ready = all(summary[a]["pilot_done"] == summary[a]["pilot_total"] for a in ANNOTATORS
                      if summary.get(a, {}).get("status") == "loaded") and len(loaded) == len(ANNOTATORS)
    if pilot_ready:
        rates = [summary[a]["pilot_rate"] for a in loaded if pd.notna(summary[a]["pilot_rate"])]
        rate = float(np.mean(rates)) if rates else float("nan")
        print(f"✅ PILOT COMPLETE — mean biased rate: {rate:.1%}")
        if rate < MIN_BIASED_RATE_PILOT:
            print(f"⚠ Rate < {MIN_BIASED_RATE_PILOT:.0%}: HINMIX annotation pool ke liye suitable nahi.")
            print("  Aage annotate karne se pehle news-based pool / MBIB label-projection pe socho.")
        else:
            print("  Pool workable hai — baaki rows annotate karo.")
    else:
        print("⏳ PILOT INCOMPLETE — pehle dono annotators ki 100 pilot rows poori karwao.")

    if all(summary.get(a, {}).get("all_done") and summary[a].get("clean") for a in ANNOTATORS):
        print("✅ FULL ANNOTATION COMPLETE & CLEAN — Phase 2 (IAA + final dataset) chalao.")
    else:
        print("⏳ Full annotation pending ya cleanup chahiye — upar ke ⚠/❌ items fix karo.")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 05
# Cell: Annotation Workspace Preparation
#
# Purpose:
# 1. Verify Phase-1 annotation sheets
# 2. Create clean working copies for A/B
# 3. Separate 100 pilot rows
# 4. Show exactly what needs to be manually annotated
# 5. NO labels are automatically assigned
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")

ANNOTATION_DIR = (
    BASE_DIR / "data" / "multilingual" / "annotation"
)

SHEETS_DIR = ANNOTATION_DIR / "sheets"
WORK_DIR = ANNOTATION_DIR / "working"

WORK_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH = ANNOTATION_DIR / "assignment_master.csv"

print("=" * 70)
print("BIASCOPE — ANNOTATION WORKSPACE")
print("=" * 70)

# ------------------------------------------------------------
# Load assignment master
# ------------------------------------------------------------

assert MASTER_PATH.exists(), (
    "assignment_master.csv missing. "
    "Run Phase 1 assignment generation first."
)

master = pd.read_csv(
    MASTER_PATH,
    encoding="utf-8"
)

print("\nAssignment master loaded:", master.shape)

# ------------------------------------------------------------
# Process both annotators
# ------------------------------------------------------------

for annotator in ["A", "B"]:

    source = SHEETS_DIR / f"annotator_{annotator}.csv"

    assert source.exists(), (
        f"Missing annotation sheet: {source}"
    )

    df = pd.read_csv(
        source,
        encoding="utf-8-sig"
    )

    # Expected assignment information
    expected = master[
        master["annotators"]
        .astype(str)
        .str.split("|")
        .map(lambda x: annotator in x)
    ].copy()

    pilot_ids = set(
        expected.loc[
            expected["role"] == "pilot",
            "row_id"
        ]
    )

    # Add annotation metadata for convenience
    df["annotation_role"] = df["row_id"].isin(pilot_ids).map(
        {True: "PILOT", False: "MAIN"}
    )

    # Keep original order and add a human-friendly index
    df.insert(
        0,
        "annotation_index",
        np.arange(1, len(df) + 1)
    )

    # Save working copy
    output = (
        WORK_DIR /
        f"annotator_{annotator}_working.csv"
    )

    df.to_csv(
        output,
        index=False,
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print(f"\n{'-' * 70}")
    print(f"ANNOTATOR {annotator}")
    print(f"{'-' * 70}")

    print("Total assigned :", len(df))
    print("Pilot rows     :", len(pilot_ids))
    print(
        "Main rows      :",
        len(df) - len(pilot_ids)
    )

    print("Working file   :", output)

    # Show first pilot examples
    pilot_preview = df[
        df["annotation_role"] == "PILOT"
    ][
        [
            "annotation_index",
            "row_id",
            "text_hi",
            "text_en",
            "bias_label",
            "subjectivity",
            "sensationalism",
            "loaded_language",
            "political_framing",
            "emotional_tone",
            "biased_span",
            "usable",
            "confidence",
            "notes"
        ]
    ].head(5)

    print("\nFirst 5 PILOT rows:")
    display(pilot_preview)

# ------------------------------------------------------------
# Annotation instructions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ANNOTATION INSTRUCTIONS")
print("=" * 70)

print("""
For every assigned sentence:

bias_label:
    0 = Neutral
    1 = Biased

Dimensions:
    0 = Absent
    1 = Present

subjectivity
sensationalism
loaded_language
political_framing
emotional_tone

usable:
    1 = Usable for bias annotation
    0 = Not usable / problematic

confidence:
    1 = Low
    2 = Medium
    3 = High

biased_span:
    Exact biased phrase/span from the Hindi sentence,
    when bias_label = 1.

notes:
    Optional explanation for difficult cases.

IMPORTANT:
Do NOT infer bias merely because the sentence:
- is political,
- discusses a controversial topic,
- contains English words,
- is negative,
- is emotional because of a factual event.

Judge the linguistic framing.
""")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("=" * 70)
print("WORKSPACE READY")
print("=" * 70)

print("\nFiles created:")

for annotator in ["A", "B"]:
    print(
        WORK_DIR /
        f"annotator_{annotator}_working.csv"
    )

print("""
NEXT STEP:
Manually annotate the 100 PILOT rows for BOTH annotators.

After pilot annotation:
    → copy completed files into:
      data/multilingual/annotation/completed/

    → name them exactly:
      annotator_A.csv
      annotator_B.csv

    → then rerun the Pre-flight Validation cell.

Only after both pilots are complete will we calculate
pilot agreement and decide whether to continue.
""")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 05: Annotation Workspace (CORRECTED)
# - Existing working files kabhi overwrite nahi hoti
# - FORCE_REBUILD=True ho to pehle backup banta hai
# - Labels automatically assign NAHI hote
# ============================================================
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
ANNOTATION_DIR = BASE_DIR / "data" / "multilingual" / "annotation"
SHEETS_DIR = ANNOTATION_DIR / "sheets"
WORK_DIR = ANNOTATION_DIR / "working"
BACKUP_DIR = ANNOTATION_DIR / "backups"
MASTER_PATH = ANNOTATION_DIR / "assignment_master.csv"
GUIDELINES_PATH = ANNOTATION_DIR / "annotation_guidelines.txt"
ADDENDUM_PATH = ANNOTATION_DIR / "guidelines_addendum.txt"

ANNOTATORS = ["A", "B"]
FORCE_REBUILD = False   # True sirf tab, jab sach me blank working file dobara chahiye (backup ban jaayega)

for d in (WORK_DIR, BACKUP_DIR):
    d.mkdir(parents=True, exist_ok=True)

SHOW_COLS = ["annotation_index", "row_id", "text_hi", "text_en", "bias_label",
             "subjectivity", "sensationalism", "loaded_language",
             "political_framing", "emotional_tone", "biased_span",
             "usable", "confidence", "notes"]

print("=" * 70)
print("BIASCOPE — ANNOTATION WORKSPACE")
print("=" * 70)

assert MASTER_PATH.exists(), "assignment_master.csv missing — pehle Phase 1 chalao"
master = pd.read_csv(MASTER_PATH, encoding="utf-8")
print("Assignment master loaded:", master.shape)

for a in ANNOTATORS:
    print(f"\n{'-' * 70}\nANNOTATOR {a}\n{'-' * 70}")

    source = SHEETS_DIR / f"annotator_{a}.csv"
    output = WORK_DIR / f"annotator_{a}_working.csv"
    assert source.exists(), f"Missing annotation sheet: {source}"

    # ---------- guard: existing working file ----------
    if output.exists():
        existing = pd.read_csv(output, encoding="utf-8-sig")
        done = int(pd.to_numeric(existing["bias_label"], errors="coerce").isin([0, 1]).sum())
        print(f"Working file already exists -> NOT overwritten.")
        print(f"Rows: {len(existing)} | labelled so far: {done}")

        if not FORCE_REBUILD:
            continue

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup = BACKUP_DIR / f"annotator_{a}_working_{stamp}.csv"
        shutil.copy2(output, backup)
        print(f"FORCE_REBUILD=True -> backup saved: {backup}")

    # ---------- build working copy ----------
    df = pd.read_csv(source, encoding="utf-8-sig")

    expected = master[master["annotators"].astype(str).str.split("|").map(lambda x: a in x)]
    pilot_ids = set(expected.loc[expected["role"] == "pilot", "row_id"])

    # sanity: sheet vs assignment
    assert set(df["row_id"]) == set(expected["row_id"]), \
        f"Annotator {a}: sheet ke row_ids assignment_master se match nahi karte"
    assert df["row_id"].is_unique, f"Annotator {a}: duplicate row_ids"

    df["annotation_role"] = np.where(df["row_id"].isin(pilot_ids), "PILOT", "MAIN")
    df.insert(0, "annotation_index", np.arange(1, len(df) + 1))
    df.to_csv(output, index=False, encoding="utf-8-sig")

    print("Total assigned :", len(df))
    print("Pilot rows     :", len(pilot_ids))
    print("Main rows      :", len(df) - len(pilot_ids))
    print("Working file   :", output)

    print("\nFirst 5 PILOT rows:")
    display(df[df["annotation_role"] == "PILOT"][SHOW_COLS].head(5))

# ---------- guidelines: source of truth = annotation_guidelines.txt ----------
print("\n" + "=" * 70)
print("ANNOTATION GUIDELINES (source of truth)")
print("=" * 70)

if GUIDELINES_PATH.exists():
    print(GUIDELINES_PATH.read_text(encoding="utf-8"))
else:
    print("annotation_guidelines.txt nahi mila — Phase 1 cell dobara chalao (assignment freeze rehta hai).")

ADDENDUM = """
GUIDELINES ADDENDUM (pilot se pehle fix kiya gaya rule)
=======================================================
1. Evaluative adjectives ko tabhi BIASED maano jab wo kisi person / group / party /
   institution ke baare me persuasive ya one-sided framing de.
   General descriptive ya encyclopedic claims (jaise bhasha, jeev-vigyan, itihas ke
   descriptions) sirf isliye biased nahi hain ki unme 'bahut', 'sashakt' jaise words hain.
2. Sentence agar fragment / UI string / numbered clause / meaningless hai -> usable = 0.
3. bias_label = 0 ho to saari dimensions 0 rakho.
4. Label Hindi text (text_hi) pe do. text_en sirf reference hai.
5. Pilot ke dauran A aur B ek dusre se discuss na karein, na labels dekhein.
"""
ADDENDUM_PATH.write_text(ADDENDUM, encoding="utf-8")
print(ADDENDUM)
print("Addendum saved:", ADDENDUM_PATH)

# ---------- next steps ----------
print("=" * 70)
print("WORKSPACE READY")
print("=" * 70)
print("""
NEXT STEPS:
1. working/annotator_<ID>_working.csv ko Google Sheets me import karo
   (har annotator sirf apni sheet dekhe).
2. Pehle sirf 100 PILOT rows independently annotate karo.
3. File -> Download -> CSV, aur completed/annotator_A.csv / annotator_B.csv
   naam se Drive me daalo.
4. Pre-flight Validation cell chalao (pilot rate + raw agreement).
5. Uske BAAD hi disagreements discuss karo, pehle nahi.
""")